# Multi-Agent Financial Analysis System

AAI-520 Final Team Project

## 1. Project Overview and GitHub Repository

This notebook is the canonical project entry point. It demonstrates the reusable research workflow implemented under `src/` without redefining application logic in notebook cells.

## 2. Setup and Configuration

This section locates the repository, imports the public workflow interface, and sets the ticker. It does not contact Yahoo Finance or Ollama; those calls begin in the end-to-end section.

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import JSON, Markdown, display


def find_project_root(start: Path) -> Path:
    """Find the repository whether Jupyter starts at its root or notebooks/."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.graph import build_research_workflow
from src.reporting import render_console_summary

workflow = build_research_workflow(project_root=PROJECT_ROOT)
TICKER = os.getenv("DEFAULT_TICKER", "AAPL").strip().upper()

print(f"Project root: {PROJECT_ROOT}")
print(f"Research ticker: {TICKER}")

## 3. Agent Design and Shared State

`ResearchWorkflow` coordinates planner, evaluator, synthesis, and memory components. Each stage adds a typed artifact to `ResearchState`, allowing the notebook and tests to inspect the same results.

## 4. Agent Functions, Data Sources, and Tool Use

The planner chooses from the registered tools. The executor invokes provider-independent tools, currently backed by the Yahoo Finance adapter, and records failures without stopping unrelated tool calls.

## 5. Workflow 1 — Prompt Chaining

The implemented research chain passes structured artifacts through planning, collection, deterministic validation, reflection, synthesis, report validation, and memory curation. The news-specific ingest-to-summary pipeline remains a separate extension point under `src/workflows/`.

## 6. Workflow 2 — Routing

Tool selection is allow-listed against the registry before execution. Specialist content routing can be added behind the existing router interface without changing this notebook entry point.

## 7. Workflow 3 — Evaluator–Optimizer

The evaluator combines deterministic checks with an LLM quality reflection, then validates the synthesized report for coverage and unsafe language. The optimizer module is scaffolded for a future feedback-driven revision loop.

## 8. Memory and Learning Across Runs

After a successful run, the memory curator stores a short timestamped lesson. The planner can retrieve recent notes for the same symbol, but current tool data always remains the source of market evidence.

## 9. End-to-End Investment Research Example

The next cell is the project's live execution point. It requires a running Ollama service with the configured model and network access for Yahoo Finance. Change `TICKER` above, then run this section.

In [ ]:
result = workflow.run(TICKER, progress=print)

### Inspect Structured Workflow Artifacts

In [ ]:
print(render_console_summary(result))
display(JSON({
    "plan": result["plan"],
    "validation": result["validation"],
    "reflection": result["reflection"],
    "report_validation": result["report_validation"],
    "memory_entry": result.get("memory_entry"),
}))

### Final Research Report

In [ ]:
display(Markdown(result["report"]))

## 10. Evaluation, Limitations, and Conclusions

The workflow surfaces deterministic validation issues and model-generated quality feedback for inspection. Current limitations include reliance on one implemented market-data provider, a local model, and a single-pass report; news routing and iterative optimization remain future extensions. Outputs are research support, not personalized investment advice.